# Auto-anonymize Subject17 with sanitized texture JPG

Non-interactive companion to `48_manual_5pt_anonymization.ipynb`. Loads the
cached 5-landmark TSV for Subject17, runs the same geometric pipeline, and
saves an `.obj` + `.mtl` + sanitized `.jpg` + `_landmarks.tsv` bundle so
downstream viewers render the mesh with full texture-resolution color.

The JPG is rebuilt from the original Einstar texture keeping only pixels
covered by the anonymized mesh's UV triangles; face-region pixels are set
to black, so opening the JPG alone reveals no face.


In [9]:
import logging
import os

import numpy as np
import pyvista as pv
import trimesh
from PIL import Image

import cedalion
import cedalion.dataclasses as cdc
import cedalion.io
from cedalion.geometry.photogrammetry.anonymization import (
    normalize_axes,
    isolate_head,
    align_axes_from_landmarks,
    detect_cap_boundary,
    face_mask_from_landmarks,
    delete_masked_vertices,
    revert_to_einstar_frame,
    save_anonymized_scan,
)

logging.getLogger('cedalion').setLevel(logging.WARNING)
pv.set_jupyter_backend('server')

# === CONFIG ===
SUBJECT_NUMBER = 21
SCANS_FOLDER = '/home/ma7/BA/PG_Subjects'
SUBJECT_DIR = f'{SCANS_FOLDER}/Subject{SUBJECT_NUMBER}'
OBJ_PATH = f'{SUBJECT_DIR}/Subject{SUBJECT_NUMBER}.obj'
LANDMARKS_TSV = f'{SUBJECT_DIR}/Subject{SUBJECT_NUMBER}_anon_landmarks.tsv'
OUT_OBJ = f'{SUBJECT_DIR}/Subject{SUBJECT_NUMBER}_anon.obj'

# Mask tunables (mm) -- same defaults as notebook 48.
LANDMARK_KEEP_RADIUS = 8.0
EAR_DELETE_RADIUS_MM = 40.0
CAP_BAND_WIDTH_MM = 15.0
CAP_BIN_SIZE_MM = 2.0
CAP_FOOT_GRAD_THRESHOLD = 0.2


## 1. Load the Einstar scan


In [10]:
surface = cedalion.io.read_einstar_obj(OBJ_PATH)
print(f'Loaded: {surface.nvertices:,} vertices, {surface.nfaces:,} faces')

# trimesh 4.6 may put the texture image on visual.material.image rather than
# visual.image. Attach the sibling JPG if neither is populated.
def _visual_image(visual):
    img = getattr(visual, 'image', None)
    if img is None:
        mat = getattr(visual, 'material', None)
        img = getattr(mat, 'image', None) if mat is not None else None
    return img

img = _visual_image(surface.mesh.visual)
if img is None:
    jpg = OBJ_PATH.replace('.obj', '.jpg')
    uv = getattr(surface.mesh.visual, 'uv', None)
    assert os.path.exists(jpg) and uv is not None
    surface.mesh.visual = trimesh.visual.TextureVisuals(
        uv=uv, image=Image.open(jpg).convert('RGBA'),
    )
    img = _visual_image(surface.mesh.visual)
print(f'Texture attached: {img.size}')


Loaded: 494,109 vertices, 866,676 faces
Texture attached: (2140, 2040)


## 2. Load cached landmarks

`Subject17_anon_landmarks.tsv` was saved by a previous run of
notebook 48 and stores the 5 landmarks in `crs=digitized` / `units=mm`.
Labels are uppercase `LPA` / `RPA` in the TSV; `normalize_landmarks_labels`
maps them to the mixed-case `Lpa` / `Rpa` convention the pipeline expects.


In [11]:
landmarks = cedalion.io.load_tsv(LANDMARKS_TSV)
# Force Lpa/Rpa casing to match the pipeline's label lookups in notebook 48.
rename_map = {l: {'LPA': 'Lpa', 'RPA': 'Rpa'}.get(l, l)
              for l in landmarks['label'].values}
if any(old != new for old, new in rename_map.items()):
    landmarks = landmarks.points.rename(rename_map)
if not hasattr(landmarks.pint, 'units') or landmarks.pint.units is None:
    landmarks = landmarks.pint.quantify('mm')

expected = {'Nz', 'Iz', 'Cz', 'Lpa', 'Rpa'}
got = set(landmarks['label'].values)
assert got == expected, f'Unexpected labels: {got}'
display(landmarks)


Magnitude,[[218.866728 145.345019 420.126304] [75.969413 232.298261 487.457971] [117.713964 128.333324 553.792396] [120.415522 17.95225 483.320784] [61.308213 95.878507 404.84577]]
Units,millimeter


## 3. Run the anonymization pipeline

Verbatim copy of notebook 48 cell `09186e9d`. All geometric work happens
here: normalize axes, isolate head, align to CTF, build the deletion mask,
preserve landmark spheres + a midline nasion strip, delete masked verts.


In [12]:
lm_raw = landmarks.pint.dequantify().values
idx = {lbl: i for i, lbl in enumerate(landmarks['label'].values)}
Nz_raw = lm_raw[idx['Nz']]

surface_n, Nz_n, R = normalize_axes(surface, Nz_raw)
landmarks_n = landmarks.pint.dequantify().copy(data=lm_raw @ R.T).pint.quantify()
surface_n, _ = isolate_head(surface_n, Nz_n)

surface_h, landmarks_n, M_ctf = align_axes_from_landmarks(surface_n, landmarks_n)
lm_n = landmarks_n.pint.dequantify().values
Nz, Iz, Cz, Lpa, Rpa = (lm_n[idx[lbl]] for lbl in ['Nz', 'Iz', 'Cz', 'Lpa', 'Rpa'])
ear_mid = 0.5 * (Lpa + Rpa)

verts = np.asarray(surface_h.mesh.vertices)
mid_y = 0.5 * (Lpa[1] + Rpa[1])
cap_z, *_ = detect_cap_boundary(
    verts, Nz, Cz, ear_mid, mid_y,
    band_width=CAP_BAND_WIDTH_MM,
    bin_size=CAP_BIN_SIZE_MM,
    foot_grad_threshold=CAP_FOOT_GRAD_THRESHOLD,
)
mask, _ = face_mask_from_landmarks(
    verts, Nz, Iz, Cz, Lpa, Rpa,
    cap_z=cap_z,
    ear_delete_radius=EAR_DELETE_RADIUS_MM,
)

for lm in (Nz, Iz, Cz, Lpa, Rpa):
    near = np.linalg.norm(verts - lm, axis=1) < LANDMARK_KEEP_RADIUS
    mask[near] = False
nasion_strip = (
    (verts[:, 2] >= Nz[2]) &
    (verts[:, 2] < cap_z) &
    (np.abs(verts[:, 1] - Nz[1]) < LANDMARK_KEEP_RADIUS) &
    (verts[:, 0] > ear_mid[0])
)
mask[nasion_strip] = False

surface_anon = delete_masked_vertices(surface_h, mask)
print(f'S{SUBJECT_NUMBER}: {surface_h.nvertices:,} -> {surface_anon.nvertices:,} verts '
      f'(-{surface_h.nvertices - surface_anon.nvertices:,}, cap_z={cap_z:.1f} mm)')


S21: 494,109 -> 378,052 verts (-116,057, cap_z=39.0 mm)


## 4. Revert to the Einstar frame

Inverts both transforms so the saved files are in the raw Einstar
(digitized) frame, matching `read_einstar_obj`'s convention.


In [13]:
surface_anon_dig, landmarks_dig = revert_to_einstar_frame(
    surface_anon, landmarks_n, R, M_ctf,
)


## 5. Save OBJ + MTL + sanitized JPG + landmarks TSV


In [14]:
written = save_anonymized_scan(surface_anon_dig, OUT_OBJ, landmarks=landmarks_dig)
for p in written:
    print(f'  {os.path.basename(p):40s} {os.path.getsize(p):>12,} bytes')


  Subject21_anon.jpg                          1,238,401 bytes
  Subject21_anon.mtl                                104 bytes
  Subject21_anon.obj                         52,335,225 bytes
  Subject21_anon_landmarks.tsv                      319 bytes


## 6. Verify the saved bundle

- Reload the saved OBJ with trimesh and confirm the texture is attached.
- Open the saved JPG directly: the face region must be pure fill (black),
  and only UV-referenced regions must carry original-texture color.
- Render the reloaded mesh with PyVista to eye-check colors.


In [15]:
reloaded = trimesh.load(OUT_OBJ, process=False)
r_img = _visual_image(reloaded.visual)
r_uv = getattr(reloaded.visual, 'uv', None)
print(f'Reloaded: {len(reloaded.vertices):,} verts, {len(reloaded.faces):,} faces')
print(f'Texture attached: {r_img is not None} size={r_img.size if r_img else None}')
print(f'UV attached: {r_uv is not None} len={len(r_uv) if r_uv is not None else None}')
assert r_img is not None and r_uv is not None
assert len(r_uv) == len(reloaded.vertices)

jpg_path = OUT_OBJ.replace('.obj', '.jpg')
arr = np.asarray(Image.open(jpg_path).convert('RGB'))
non_fill = np.any(arr != 0, axis=-1)
print(f'JPG non-fill pixels: {non_fill.mean():.1%} of {arr.shape[0]*arr.shape[1]:,}')


Reloaded: 378,052 verts, 652,655 faces
Texture attached: True size=(2140, 2040)
UV attached: True len=378052
JPG non-fill pixels: 55.5% of 4,365,600


In [16]:
# Eye-check: bake texture -> per-vertex RGB and feed explicitly to PyVista
# (pv.wrap(trimesh) doesn't forward TextureVisuals).
colored = reloaded.copy()
colored.visual = reloaded.visual.to_color()
pv_mesh = pv.wrap(colored)
pv_mesh['rgb'] = np.asarray(colored.visual.vertex_colors)[:, :3]
pvplt = pv.Plotter()
pvplt.add_mesh(pv_mesh, scalars='rgb', rgb=True, smooth_shading=True)
pvplt.add_text(
    f'S{SUBJECT_NUMBER} anonymized (OBJ+MTL+sanitized JPG)',
    position='upper_left', font_size=12,
)
pvplt.show()


Widget(value='<iframe src="http://localhost:43197/index.html?ui=P_0x7fed1a5f0710_1&reconnect=auto" class="pyvi…